In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:
import sys
import os
import importlib.util

# 1. Define the path to the models file
# Assuming you are in the 'notebooks' folder or similar, we go up one level to 'src'
path_to_models = os.path.abspath(os.path.join('..', 'src', 'models.py'))

# 2. Load the module specifically
spec = importlib.util.spec_from_file_location("models_direct", path_to_models)
models_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(models_module)

# 3. Pull the functions you need
build_model = models_module.build_model
GATv2_Model = models_module.GATv2_Model

print("✅ Success! Models loaded by bypassing src/__init__.py")

✅ Success! Models loaded by bypassing src/__init__.py


In [5]:
# 1. Generate the Dictionary Data (50 nodes, 2D Q/K/V)
def get_data(n=50):
    q, k, v = torch.randn(n, 2), torch.randn(n, 2), torch.randn(n, 2)
    x = torch.cat([q, k, v], dim=1) 
    adj = torch.ones(n, n) # Fully connected
    labels = torch.tensor([torch.argmin(torch.norm(k - q[i], dim=1)) for i in range(n)])
    return x, adj, labels

x, adj, labels = get_data()

In [6]:
# 1. Initialize the standard GAT model
# We use the same hyperparameters (32 hidden, 4 heads) for a fair comparison
model_gat = build_model("gat", nfeat=6, nhid=32, nclass=50, nheads=4, dropout=0.1, alpha=0.2)
optimizer_gat = torch.optim.Adam(model_gat.parameters(), lr=0.01)

print(f"{'Epoch':<8} | {'Loss':<8} | {'Acc':<8} ")
print("-" * 40)

for epoch in range(1001):
    model_gat.train()
    optimizer_gat.zero_grad()
    
    # Forward pass
    out = model_gat(x, adj)
    loss = F.nll_loss(out, labels)
    
    # Backward pass
    loss.backward()
    optimizer_gat.step()
    
    if epoch % 100 == 0:
        model_gat.eval()
        with torch.no_grad():
            pred = model_gat(x, adj)
            acc = (pred.argmax(1) == labels).float().mean().item()
            print(f"{epoch:<8} | {loss.item():<8.4f} | {acc:<8.2%}")

print("-" * 50)
print(f"GAT Final Accuracy: {acc:.2%}")

Epoch    | Loss     | Acc      
----------------------------------------
0        | 3.9962   | 8.00%   
100      | 3.3774   | 8.00%   
200      | 3.3722   | 8.00%   
300      | 3.3752   | 8.00%   
400      | 3.3812   | 8.00%   
500      | 3.3755   | 8.00%   
600      | 3.3813   | 8.00%   
700      | 3.3775   | 8.00%   
800      | 3.3735   | 8.00%   
900      | 3.3776   | 8.00%   
1000     | 3.3767   | 8.00%   
--------------------------------------------------
GAT Final Accuracy: 8.00%


In [7]:
import numpy as np

# Re-initialize for a fresh tracked run
model_v2 = GATv2_Model(nfeat=6, nhid=32, nclass=50, nheads=4, dropout=0.1, alpha=0.2)
optimizer = torch.optim.Adam(model_v2.parameters(), lr=0.01)

print(f"{'Epoch':<8} | {'Loss':<8} | {'Acc':<8}")
print("-" * 40)

for epoch in range(1501):
    model_v2.train()
    optimizer.zero_grad()
    
    out = model_v2(x, adj)
    loss = F.nll_loss(out, labels)
    loss.backward()
    optimizer.step()
    
    if epoch % 100 == 0 or epoch == 1500:
        model_v2.eval()
        with torch.no_grad():
            # Get the output and calculate metrics
            current_out = model_v2(x, adj)
            acc = (current_out.argmax(1) == labels).float().mean().item()
            print(f"{epoch:<8} | {loss.item():<8.4f} | {acc:<8.2%} |")

        if acc >= 1.0:
            print("-" * 60)
            print(f"100% Accuracy reached at epoch {epoch}!")
            break

Epoch    | Loss     | Acc     
----------------------------------------
0        | 4.0678   | 8.00%    |
100      | 1.7514   | 66.00%   |
200      | 1.4084   | 78.00%   |
300      | 1.3186   | 82.00%   |
400      | 0.9055   | 92.00%   |
500      | 0.9877   | 94.00%   |
600      | 1.0324   | 96.00%   |
700      | 0.7816   | 96.00%   |
800      | 0.7579   | 96.00%   |
900      | 0.9593   | 92.00%   |
1000     | 0.6813   | 98.00%   |
1100     | 0.9558   | 98.00%   |
1200     | 1.0807   | 92.00%   |
1300     | 0.6386   | 98.00%   |
1400     | 0.7118   | 100.00%  |
------------------------------------------------------------
100% Accuracy reached at epoch 1400!
